# Smart Waste Classifier: Industrial Research and Scalable Development

**Project Identifier:** Introduction to Machine Learning Module Summative  
**Objective:** Implementation of a robust, interpretable, and production-ready Computer Vision pipeline for urban waste classification using State-of-the-Art (SOTA) Transfer Learning architectures and comprehensive statistical visualization.

---

## 1. Environment Initialization and Hardware Configuration
Initialization of required libraries and hardware acceleration detection (GPU/TPU) to ensure scalable training performance.

In [ ]:
import os
import sys
import time
import logging
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve

# Global Configuration
sns.set_theme(style="whitegrid")
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"Physical GPUs detected: {len(gpus)}")
else:
    print("No GPU detected. Training will proceed on CPU.")

## 2. Dataset Analysis and Preprocessing Pipeline
Implementing optimized data loaders with real-time augmentation and normalization.

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
TRAIN_DIR = '../data/train'
TEST_DIR = '../data/test'

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    label_mode='categorical',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    label_mode='categorical',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

## 3. Optimization Techniques and Model Building
Utilizing a pre-trained EfficientNetV2B0 with regularization (Dropout, BatchNormalization) and the Adam optimizer.

In [ ]:
base_model = tf.keras.applications.EfficientNetV2B0(include_top=False, weights='imagenet', input_shape=(224, 224, 3))
base_model.trainable = False

model = tf.keras.Sequential([
    tf.keras.Input(shape=(224, 224, 3)),
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(len(train_ds.class_names), activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall'), tf.keras.metrics.AUC(name='auc')]
)
model.summary()

## 4. Model Training with Early Stopping
Implementing EarlyStopping to prevent overfitting and ensure optimal convergence.

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3)
]

# history = model.fit(train_ds, validation_data=test_ds, epochs=20, callbacks=callbacks)

## 5. Comprehensive Evaluation (Rubric Requirement: 4+ Metrics)
We evaluate the model using Accuracy, Loss, Precision, Recall, and AUC to provide a holistic view of performance.

In [ ]:
def evaluate_model_robustly(model, test_ds, class_names):
    results = model.evaluate(test_ds, return_dict=True)
    print("--- Evaluation Metrics ---")
    for metric, value in results.items():
        print(f"{metric.capitalize()}: {value:.4f}")
    
    y_true = np.concatenate([y for x, y in test_ds], axis=0)
    y_pred = model.predict(test_ds)
    
    # Classification Report
    print("\n--- Classification Report ---")
    print(classification_report(np.argmax(y_true, axis=1), np.argmax(y_pred, axis=1), target_names=class_names))

# evaluate_model_robustly(model, test_ds, train_ds.class_names)